# 05_scans_and_controls: independent resumable A100 bundle

Two-construction pure S2 entanglement-transition scan, max-mix S2 arm, and M2–M3 controls. This notebook launches only `05_scans_and_controls`; it never continues into another experiment. Every ordinary stochastic case uses 10 independent trajectories in two immutable five-trajectory shards. Run the default report-only pass first, inspect the exact verified/pending inventory, then set `RESUME_REPORT_ONLY=False` to compute only missing shards.

## 1. Mount Drive and locate the campaign package


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

MYDRIVE = Path('/content/drive/MyDrive')
CAMPAIGN_ROOT = MYDRIVE / 'final_production_ready_figure_scripts'
runner = CAMPAIGN_ROOT / 'colab_bundle_runner.py'
bundle_root = CAMPAIGN_ROOT / 'prior_designs' / '05_scans_and_controls'
if not runner.is_file() or not (bundle_root / 'run_bundle.py').is_file():
    raise FileNotFoundError('Upload the complete updated final_production_ready_figure_scripts folder')
bundle_config = json.loads((bundle_root / 'production_config.json').read_text())
OUTPUT_BUNDLE = bundle_config.get('output_bundle', bundle_root.name)
print(f'[campaign] {CAMPAIGN_ROOT}')
print(f'[bundle] {bundle_root}')


## 2. Select this bundle's queue


In [ ]:
RUN_PROFILE = 'production'
BUNDLE = '05_scans_and_controls'
CASE_PREFIXES = ()  # e.g. ('S2_',); empty runs every currently eligible case
CASE_IDS = ()       # exact IDs; empty uses prefixes/full bundle
MAX_SESSION_GPU_HOURS_OVERRIDE = None
PREFLIGHT_ONLY = False
RESUME_REPORT_ONLY = True  # inspect first; change to False to launch missing work
HEARTBEAT_SECONDS = 60
RUN_QUEUE = True
if HEARTBEAT_SECONDS <= 0:
    raise ValueError('HEARTBEAT_SECONDS must be positive')
if CASE_PREFIXES and CASE_IDS:
    raise ValueError('use CASE_PREFIXES or CASE_IDS, not both')
print(json.dumps({
    'bundle': BUNDLE, 'profile': RUN_PROFILE, 'case_prefixes': CASE_PREFIXES,
    'case_ids': CASE_IDS, 'resume_report_only': RESUME_REPORT_ONLY,
    'heartbeat_seconds': HEARTBEAT_SECONDS,
}, indent=2))


## 3. Verify, resume, and stop at the end of this bundle


In [ ]:
if not RUN_QUEUE:
    raise RuntimeError('RUN_QUEUE is False')
command = [
    sys.executable, '-u', str(runner), '--drive-root', str(MYDRIVE),
    '--bundle', BUNDLE, '--profile', RUN_PROFILE,
    '--heartbeat-seconds', str(HEARTBEAT_SECONDS),
]
for prefix in CASE_PREFIXES:
    command += ['--case-prefix', str(prefix)]
for case_id in CASE_IDS:
    command += ['--case-id', str(case_id)]
if MAX_SESSION_GPU_HOURS_OVERRIDE is not None:
    command += ['--max-session-hours', str(MAX_SESSION_GPU_HOURS_OVERRIDE)]
if PREFLIGHT_ONLY:
    command.append('--preflight-only')
if RESUME_REPORT_ONLY:
    command.append('--resume-report-only')
production_collection = 'classA_final_production_outputs/production_10sample_v4_occupied_frame_cycle_resolved'
pilot_collection = 'classA_pilot_outputs/production_10sample_v4_occupied_frame_cycle_resolved'
collection = production_collection if RUN_PROFILE == 'production' else pilot_collection
output_root = MYDRIVE / collection
session_dir = output_root / '_bundle_sessions'
build_id = hashlib.sha256(runner.read_bytes()).hexdigest()[:16]
print('[bundle dashboard]')
print(json.dumps({
    'bundle': BUNDLE, 'profile': RUN_PROFILE, 'output_root': str(output_root),
    'runner_build_id': build_id, 'heartbeat_seconds': HEARTBEAT_SECONDS,
    'session_dir': str(session_dir), 'case_prefixes': CASE_PREFIXES,
    'case_ids': CASE_IDS, 'resume_report_only': RESUME_REPORT_ONLY,
}, indent=2))
print(f'[launch bundle] {" ".join(command)}', flush=True)
result = subprocess.run(command, check=False)
if result.returncode:
    sessions = sorted(session_dir.glob(f'*_{BUNDLE}_*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if sessions:
        latest = json.loads(sessions[0].read_text())
        print('[saved bundle failure]')
        print(json.dumps({'session': str(sessions[0]), 'failure': latest.get('failure'), 'current': latest.get('current'), 'log': latest.get('session_log')}, indent=2))
    raise subprocess.CalledProcessError(result.returncode, command)


## 4. Analyze the completed 10-trajectory pure S2 matrix


In [ ]:
RUN_S2_ENTANGLEMENT_ANALYSIS = False
if RUN_S2_ENTANGLEMENT_ANALYSIS:
    analysis_root = output_root / OUTPUT_BUNDLE / 'S2_entanglement_analysis'
    archive_roots = [
        output_root / OUTPUT_BUNDLE,
        MYDRIVE / 'classA_final_production_outputs' / BUNDLE,
    ]
    command = [sys.executable, '-u', str(bundle_root / 'src' / 's2_entanglement_analysis.py'), '--bundle-root', str(bundle_root), '--output-root', str(analysis_root)]
    for archive_root in archive_roots:
        command += ['--archive-root', str(archive_root)]
    subprocess.run(command, check=True)
    print((analysis_root / 's2_entanglement_report.json').read_text())


## Final. Disconnect this completed Colab runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
